In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path_to_capstone_folder = "/content/drive/Othercomputers/My Laptop/Capstone_Fraud_Detection"

In [ ]:
# Point this at the folder where you uploaded/synced the whole project in Drive
%cd $path_to_capstone_folder
!pip install -q -r fraud_triage_agent/requirements.txt


/content/drive/Othercomputers/My Laptop/Capstone_Fraud_Detection


In [4]:
!pytest fraud_triage_agent/tests -q
from fraud_triage_agent.tools.data_loader import high_risk
from fraud_triage_agent.agents.orchestrator import triage
tid = str(int(high_risk().TransactionID.iloc[0]))
print(triage(tid).model_dump_json(indent=2))

.......                                                                  [100%]
7 passed in 5.52s
{
  "transaction_id": 3663769,
  "recommendation": "Likely fraud, review recommended",
  "summary": "Transaction 3663769 has a model fraud probability of 94%. Top SHAP drivers by absolute contribution: Count_C1 (increases risk, SHAP +1.5345), PaymentCard_Dummy1 (increases risk, SHAP +0.4898), Count_C5 (increases risk, SHAP +0.3441), Log_TransactionAmt (decreases risk, SHAP -0.2184), Days_Since_D2 (increases risk, SHAP +0.2069). Among 20 similar validation transactions, 16 were TP and 4 were FP (TP share among TP/FP: 80%; 95% Wilson interval 58%–92%). Overall model drift status: significant shift. None of this transaction's top drivers show drift (all stable). Most drifted features overall: GNN_Emb_8, GNN_Emb_10, GNN_Emb_7.",
  "interpretation": "The SHAP analysis indicates that several features, particularly Count_C1, PaymentCard_Dummy1, Count_C5, and Days_Since_D2, are driving the risk sc

In [ ]:
%cd $path_to_capstone_folder
!python -m fraud_triage_agent.evals.run_evals

/content/drive/Othercomputers/My Laptop/Capstone_Fraud_Detection
{
  "invariants": {
    "n_transactions": 12155,
    "pass_rates": {
      "five_features": 1.0,
      "sorted_by_abs_shap": 1.0,
      "probability_matches_csv": 1.0,
      "direction_matches_sign": 1.0,
      "history_counts_consistent": 1.0,
      "matches_within_cutoff": 1.0,
      "tp_fp_counts_match_matches": 1.0,
      "recommendation_present": 1.0
    },
    "failures": []
  },
  "retrieval_loo": {
    "n_rows": 2417,
    "tp_share": 0.582126603227141,
    "coverage": 1.0,
    "accuracy": 0.687215556474969,
    "baseline_majority_accuracy": 0.582126603227141,
    "mean_neighbours_used": 19.992552751344643
  },
  "guardrails": {
    "cases": {
      "rejects_letters": true,
      "rejects_too_short": true,
      "rejects_unknown_id": true,
      "rejects_injection_text": true,
      "rejects_percentage": true,
      "rejects_definitive_claim": true,
      "accepts_valid_id": true,
      "accepts_feature_names_with_

In [7]:
!pip install -U langchain langchain_openai langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 kB 4.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 574.7/574.7 kB 16.1 MB/s eta 0:00:0000:01
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.12
    Uninstalling langchain-core-1.6.12:
      Successfully uninstalled langchain-core-1.6.12
  Attempting uninstall: langchain
    Found existing installation: langchain 1.4.6
    Uninstalling langchain-1.4.6:
      Successfully uninstalled langchain-1.4.6


In [18]:
!pkill -f cloudflared
!pkill -f streamlit

In [6]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [ ]:
import os, subprocess, time
ROOT = path_to_capstone_folder
env = {**os.environ, "PYTHONPATH": ROOT}
subprocess.Popen(
    ["streamlit", "run", "fraud_triage_agent/ui/streamlit_app.py",
     "--server.port", "8501", "--server.headless", "true"],
    cwd=ROOT, env=env, stdout=open("/content/streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)
print(open("/content/streamlit.log").read())



2026-10-27 17:19:51.773 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.48.102.101:8501




In [8]:
import re
p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501"],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if m:
        print("Open:", m.group(0)); break

Open: https://ham-blah-background-population.trycloudflare.com
